# PDF/Layout Chunking

This is talking about a problem that traditional PDF text extraction cannot solve properly.

## The Problem with Normal PDF Chunking

Suppose you have a PDF page like this:

```text
------------------------------------------------
| Company Revenue Report 2025                  |
------------------------------------------------

Main Text Column            Chart

Revenue increased           [ Bar Chart ]
by 25% in Q1 due            [ Revenue Data ]
to strong sales...

Revenue in Q2...
```

A normal PDF parser (PyPDF, PDFPlumber, etc.) often extracts text in reading order like:

```text
Company Revenue Report 2025

Revenue increased by 25% in Q1 due
to strong sales...

Bar Chart Revenue Data

Revenue in Q2...
```

Or even worse:

```text
Revenue increased
Bar Chart Revenue Data
by 25% in Q1 due
Revenue in Q2...
```

The layout information is lost.

When you create embeddings from this text, the vector database thinks:

> "Bar Chart Revenue Data" belongs in the middle of the paragraph.

This creates poor retrieval quality.

---

## What Layout Chunking Tries To Do

Instead of only understanding text, the system also understands:

- Position on page
- Visual structure
- Tables
- Images
- Captions
- Sidebars
- Headers
- Footnotes

So it knows:

```text
Paragraph A → Left Column

Chart → Right Side

Caption → Below Chart

Footer → Bottom
```

and keeps them separate.

---

## Example

A research paper page:

```text
------------------------------------------------
| Title                                        |
------------------------------------------------

Paragraph 1

Paragraph 2

Figure 1: Architecture Diagram

Paragraph 3
```

### Normal Extraction

```text
Paragraph 1
Paragraph 2
Figure 1
Architecture Diagram
Paragraph 3
```

### Layout-Aware Extraction

```text
Chunk 1:
Paragraph 1 + Paragraph 2

Chunk 2:
Figure 1 + Architecture Diagram

Chunk 3:
Paragraph 3
```

Much cleaner retrieval.

---

## What is ColPali?

ColPali is a retrieval model that treats a PDF page like an image rather than plain text.

Instead of:

```text
PDF
 ↓
Extract Text
 ↓
Embedding Model
```

it does:

```text
PDF Page Screenshot
 ↓
Vision Language Model
 ↓
Visual Embedding
```

The embedding captures:

- Text content
- Spatial position
- Layout
- Tables
- Figures
- Charts

all together.

---

## Why is this Useful?

Imagine a user asks:

> Show me the chart explaining quarterly revenue.

### Traditional RAG

- Searches only text
- Might miss the chart entirely

### Layout-Aware Retrieval

- Embedding contains chart position and visual semantics
- Retrieves the actual page containing the chart

This is why ColPali became popular for document retrieval.

---

## What Actually Gets Stored?

### Traditional RAG

Chunk:

```text
Revenue increased by 25%...
```

Embedding:

```text
[0.12, 0.56, ...]
```

---

### Layout RAG

Document:

```text
Page Image
```

Embedding represents:

```text
- Revenue text
- Chart
- Caption
- Relative positions
```

So the vector represents the entire visual structure of the page.

---

## When Should You Use It?

### Use Layout/VLM Chunking For

- Research papers
- Financial reports
- Annual reports
- Medical documents
- Scanned PDFs
- Invoices
- Legal contracts
- Documents with charts/tables

---

### Don't Use It For

- CSV files
- Plain text documents
- Knowledge base articles
- FAQs
- Source code
- Structured database records

For those, standard semantic chunking is usually better and cheaper.

---

## Modern Production RAG (2026)

### 1. Structured Data

```text
CSV / SQL
    ↓
SQL Querying
```

No embeddings required.

---

### 2. Plain Text Documents

```text
Documents
    ↓
Semantic Chunking
or
Parent-Child Chunking
```

---

### 3. Complex PDFs

```text
PDF
    ↓
Layout-Aware Retrieval
(ColPali, ColQwen2, etc.)
```

Suitable for:

- Charts
- Tables
- Multi-column layouts
- Figures
- Scanned documents

---

## Key Takeaway

> Traditional embeddings understand **what the text says**.

> Layout-aware embeddings understand **what the page looks like and where everything is located**.

### Step 1: Convert PDF pages to images

In [ ]:
from pdf2image import convert_from_path

pages = convert_from_path("report.pdf")

for i, page in enumerate(pages):
    page.save(f"page_{i}.png")

### Step 2: Load ColPali

In [ ]:
from colpali_engine.models import ColPali
from transformers import AutoProcessor

model = ColPali.from_pretrained(
    "vidore/colpali-v1.2"
)

processor = AutoProcessor.from_pretrained(
    "vidore/colpali-v1.2"
)

ModuleNotFoundError: No module named 'colpali_engine'

In [ ]:
from PIL import Image
import torch

image = Image.open("page_0.png")

batch = processor(
    images=[image],
    return_tensors="pt"
)

with torch.no_grad():
    embedding = model(**batch)

### Store in Quadrant

In [ ]:
from qdrant_client import QdrantClient

client = QdrantClient("localhost", port=6333)

client.upsert(
    collection_name="pdf_pages",
    points=[
        {
            "id": 1,
            "vector": embedding.tolist(),
            "payload": {
                "page": 0,
                "file": "report.pdf"
            }
        }
    ]
)

### Querying

In [ ]:
# Step 1: Encode query
query = "Show revenue chart for Q1"

query_batch = processor(
    text=[query],
    return_tensors="pt"
)

with torch.no_grad():
    query_embedding = model(**query_batch)
# Step 2: Search
results = client.search(
    collection_name="pdf_pages",
    query_vector=query_embedding.tolist(),
    limit=5
)